In [1]:
import sys
sys.path.append("../")  # Goes up to "project/"
import test_operators as top
import test_dataloader as tdl
import test_loss as tl
import test_latticeConv as tge 
import parameters as var
var.print_parameters()

*********** Configuration parameters ***********
* β=2, Nx=32, Nt=32
* Variables=2048
* m0=-0.1884
* blocks_x=2, blocks_t=2 (for the aggregation)
* SAP vectors for the loss function Nv=30
* Fake test vectors generated Nv=10
* Number of confs=500
* Confs used for training=50
* Device: cuda:0
* Precision: double
* DOF on fine grid: 2048
* DOF on coarse grid: 80
* Gauge equivariant model: True
************************************************


# Testing operators

In [2]:
nv, nx, nt, blocks_x, blocks_t = 2, 4, 4, 2, 2
opsTest = top.TestOperators(nv,nx,nt,blocks_x,blocks_t)
opsTest.assembleP()
opsTest.assemblePdagg()
# In case I want to print the operators
#opsTest.printP()
#opsTest.printPdagg()

Nv=2, Nx=4, Nt=4, Blocks X=2, Blocks T=2
Test vectors not orthonormalized


In [3]:
#Checking P*^T = Pdagg
opsTest.testP_Pdagg()
#Checking that local orthonormalization of test vectors is correct.
opsTest.testOrthonormalization()

KeyboardInterrupt: 

# Testing dataloader

In [ ]:
dloaderTest = tdl.TestDataloader(5,"other")
dloaderTest.testDimensions()
dloaderTest.testDataType()
dloaderTest.testConfZero()
print()
dloaderTest.testTVector()
print()
dloaderTest.testPlaquette()
print()
dloaderTest.testPlaquettePt()
print()
dloaderTest.testPlaquetteDataLoader()
print("*************\nCheck H5FD\n*************")
dloaderTest = tdl.TestDataloader(5,"h5")
dloaderTest.testDimensions()
dloaderTest.testDataType()
dloaderTest.testConfZero()
print()
dloaderTest.testTVector()
print()
dloaderTest.testPlaquette()
print()
dloaderTest.testPlaquettePt()
print()
dloaderTest.testPlaquetteDataLoader()

# Testing loss function

In [ ]:
batch_size = 100
lossTest = tl.TestLoss(batch_size)
lossTest.testTrivialCase()
lossTest.testRandomCase()
lossTest.testRemainderTV()

In [ ]:
#import convert_to_pt
#convert_to_pt.binaryPlaq2ptPlaq()

In [ ]:
import sys
sys.path.append("../")  # Goes up to "project/"
import test_latticeConv as tge 
import parameters as var
import numpy as np
import torch
import torch.nn as nn

In [ ]:
#Check parallel transport
lcnn = tge.TestLCNN(100)
lcnn.check_u1_vars()
print(lcnn.w.shape)
wxkm = lcnn.check_parallel_transport()
w = lcnn.w
lcnn.print_parallel_transport()

In [ ]:
u, w = lcnn.LConv_test()
w[0,0,0,0].item()

In [ ]:
tge.test_LCNN_layers()

In [ ]:
u = torch.Tensor(100,3,3)

def transporter(u,orientation,mu,k):
    p_transporter = 1
    u_mu = u[:,mu] #(B,NT,NX)
    #x+mu
    if orientation == -1:
        #We are just multiplying U(1) numbers, so the order of operations is not relevant. For a different group this 
        #function would have to be rewritten respecting the order. 
        for i in range(k):
            p_transporter *= torch.roll(u_mu,shifts=i*orientation,dims=1+mu)
    #x-mu
    elif orientation == 1:
        for i in range(1,k+1):
            p_transporter *= torch.roll(u_mu.conj(),shifts=i*orientation,dims=1+mu)
    return p_transporter.unsqueeze(1)    

In [ ]:
u = torch.zeros([100,2,3,3],dtype=var.PREC_COMPLEX)
u[0,0,0,0] = 1j
u[0,0,0,1] = 2j
u[0,0,0,2] = 3j
u[0,0,1,0] = 4j
u[0,0,1,1] = 5j
u[0,0,1,2] = 6j
u[0,0,2,0] = 7j
u[0,0,2,1] = 8j
u[0,0,2,2] = 9j

In [ ]:
orientation = 1
mu = 0
k = 2
#product = u*torch.roll(u[:,mu],shifts=orientation,dims=1+mu).unsqueeze(1)
pt = transporter(u,orientation,mu,k)
print("Transporter function",pt[0,0])
print("Product",product[0,0])

# Testing parallel transport along arbitrary paths

In [ ]:
import numpy as np
import torch
import torch.nn as nn

In [ ]:
def Tp(u,path):
    #path = [-1,-2,-1,+2,+2] 1 = \hat{t}, 2 = \hat{x}, I don't use +0 -0 for obvious reasons
    #u.shape = (Batch,2,NT,NX)
    p_transporter = 1
    for p in path:
        mu  = abs(p)-1
        if p > 0:
            #Hp = U^+_p(x-p)
            #x-p
            #u[:,0] = torch.roll(u[:,0],shifts=1,dims=1+mu)
            #u[:,1] = torch.roll(u[:,1],shifts=1,dims=1+mu)
            u = torch.roll(u, shifts=1, dims=2+mu)
            p_transporter *= u[:,mu].conj()
        elif p < 0:
            #x+p
            p_transporter *= u[:,mu] #We roll both components ...
            #u[:,0] = torch.roll(u[:,0],shifts=-1,dims=1+mu)
            #u[:,1] = torch.roll(u[:,1],shifts=-1,dims=1+mu)
            u = torch.roll(u, shifts=-1, dims=2+mu) 
        else:
            raise("p should be positive or negative, not zero")
    return p_transporter
        

In [ ]:
path = [-1,-2,-1,2,2]
B, Nx, Nt = 100, 4, 4
# Random phases in [0, 2π)
theta = 2 * torch.pi * torch.rand(B, 2, Nt, Nx, dtype=torch.float64)
# U(1) variables with complex128 precision
u = torch.exp(1j * theta)
u_roll1 = torch.zeros(B, 2, Nt, Nx, dtype=torch.complex128)
u_roll2 = torch.zeros(B, 2, Nt, Nx, dtype=torch.complex128)
print(u.shape)   # torch.Size([B, 2, Nt, Nx])
print(u.dtype)   # torch.complex128

In [ ]:
mu = 0
u_roll1[:,0] = torch.roll(u[:,0],shifts=1,dims=1+mu)
u_roll1[:,1] = torch.roll(u[:,1],shifts=1,dims=1+mu)
u_roll2 = torch.roll(u, shifts=1, dims=2+mu)

In [ ]:
print("Original tensor")
print(u[0,0,1,0])
print(u[0,1,1,0])
print("Roll1 tensor")
print(u_roll1[0,0,0,0])
print(u_roll1[0,1,0,0])
print("Roll2 tensor")
print(u_roll2[0,0,0,0])
print(u_roll2[0,1,0,0])
print("Comparing rolled tensors")
torch.allclose(u_roll1,u_roll2)

In [ ]:
parallel_transport = Tp(u,path)
print("Parallel transport shape",parallel_transport.shape)
#Constructing the transport along [-1,-2,-1,2,2] manually.
nx, nt = 3, 2
b = 0
pt = u[b,0,nt,nx] * u[b,1,(nt+1)%Nt,nx] * u[b,0,(nt+1)%Nt,(nx+1)%Nx] * u.conj()[b,1,(nt+1+1)%Nt,(nx+1-1)%Nx] * u.conj()[b,1,(nt+1+1)%Nt,(nx+1-1-1)%Nx]
print("Manual transport",pt)
print("Using the function",parallel_transport[0])

In [ ]:
import sys
sys.path.append("../")  # Goes up to "project/"
import test_operators as top
import test_dataloader as tdl
import test_loss as tl
import test_latticeConv as tge 
import parameters as var
var.print_parameters()

#Test PTC layers
tge.test_PTC_layers()